# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wizard-i/FlyRank-Starter-Repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### 1. Unit of Analysis

The raw daily fact table contains one row per report date, client, and content page.

For my lane, the final analysis unit is one pseudonymized content page for one client, aggregated across the March 2026 feature window.

### 2. Tables I Will Use

- `fact_content_daily_performance`: March search-performance features and April outcome measurements.
- `dim_content`: content creation date, used to calculate content age at the decision cutoff.

I will not use the query-level table in this first contract.

### 3. Time Windows

- Feature window: March 1–31, 2026.
- Decision cutoff: March 31, 2026.
- Outcome window: April 1–30, 2026.

All five features must be knowable by the March 31 decision cutoff. April measurements are reserved for the outcome label.

### 4. Target / Proxy

My outcome proxy is `went_dark_apr`.

It equals 1 when a page has zero Google Search Console clicks across at least 20 observed GSC-available days in April. It equals 0 when at least one click is observed.

This measures a future no-click outcome, not whether a content refresh will cause recovery.

### 5. One Deliberate Exclusion

I deliberately exclude the entire `fact_content_query_90d` table because its fixed 90-day window does not align safely with my March feature and April outcome windows.

### Expected Human Output

A ranked review queue that helps content editors decide which pages deserve further investigation and possible refresh actions.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


%pip -q install -U duckdb

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# Read the token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add it in Colab Secrets first."
    )

# Connect DuckDB securely
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Keep the secret out of the SQL source text
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

# Warehouse paths
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

# Mid-panel feature and outcome windows
MARCH = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{FACT}/month=2026-04/*.parquet')"

# Supporting content dimension
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("DuckDB connected successfully.")
print("Feature window: March 2026")
print("Outcome window: April 2026")


DuckDB connected successfully.
Feature window: March 2026
Outcome window: April 2026


## 2. Fields: feature / label / context / excluded


### Five Features

| Feature | Meaning | Knowable at the decision moment because... |
|---|---|---|
| `impressions_mar` | Total GSC impressions in March | March search exposure was measured before the March 31 cutoff. |
| `clicks_mar` | Total GSC clicks in March | These clicks were already observed during the feature window. |
| `ctr_mar_pct` | March clicks divided by March impressions | Both measurements were available by March 31. |
| `avg_position_mar` | Impression-weighted average search position | It is calculated only from March position measurements. |
| `content_age_days_mar` | Age of the page on March 31 | It is calculated from the page's creation date and the fixed cutoff date. |

### Label / Proxy

- `went_dark_apr`: Zero observed GSC clicks across at least 20 GSC-available days in April.

### Context

- `client_hash_id`: Identifies the client for grouping and joining.
- `content_hash_id`: Identifies the content page for grouping and joining.
- `report_date`: Identifies the observation date.
- `gsc_data_available`: Identifies whether GSC data is measured.
- `gsc_days_apr`: Tracks the amount of measured April data for outcome eligibility.

These fields are not model features.

### Excluded

- `fact_content_query_90d`: excluded because its fixed 90-day measurement window overlaps the selected time windows.
- `last_optimized_date` and `has_been_optimized`: excluded because these can reflect product decisions or information unavailable at the March 31 cutoff.
- Client and content identifiers: context only, never predictive features.

### Missing Data

A row with `gsc_data_available = FALSE` is not evidence of zero search traffic.

I will only use measured GSC rows for the selected features and outcome. I will not treat unavailable GA4 measurements as zero engagement.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# BUILD APRIL OUTCOME LABEL

label_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    CAST(SUM(gsc_sum_position) AS DOUBLE)
        / NULLIF(SUM(gsc_impressions), 0)
        AS avg_gsc_position_apr,

    SUM(gsc_impressions)
        AS total_gsc_impressions_apr,

    COUNT(DISTINCT report_date)
        AS n_gsc_days_apr

FROM read_parquet('{APRIL_PATH}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(gsc_impressions) >= 100
"""

april_outcomes = con.sql(label_query).df()

print("APRIL OUTCOME DATA")
print("=" * 40)

print("April eligible page-client pairs:", len(april_outcomes))

display(april_outcomes.head(5))


NameError: name 'APRIL_PATH' is not defined

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# QUERY 1: Verify the raw daily grain

grain_query = con.sql(f"""
    SELECT COUNT(*) AS duplicate_grain_groups
    FROM (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id
        FROM {MARCH}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    ) AS duplicates
""").df()

display(grain_query)


# QUERY 2: Verify row count and date span

count_query = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date) AS distinct_days,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content_items
    FROM {MARCH}
""").df()

display(count_query)


# QUERY 3: Verify measured data availability

availability_query = con.sql(f"""
    SELECT
        'GSC available' AS data_type,
        COUNT(*) AS surviving_rows
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE

    UNION ALL

    SELECT
        'GA4 available',
        COUNT(*)
    FROM {MARCH}
    WHERE ga4_data_available IS TRUE

    UNION ALL

    SELECT
        'Both available',
        COUNT(*)
    FROM {MARCH}
    WHERE
        gsc_data_available IS TRUE
        AND ga4_data_available IS TRUE
""").df()

display(availability_query)


In [ ]:

# Match March features with April outcomes


# ==============================================
# BUILD MARCH FEATURES + MATCH APRIL OUTCOMES
# ==============================================

# Step 1: Inspect the content dimension schema.
# This avoids guessing the creation-date column name.

dim_schema = con.sql(
    f"DESCRIBE SELECT * FROM {DIM_CONTENT}"
).df()

dim_columns = dim_schema["column_name"].tolist()

if "content_hash_id" not in dim_columns:
    raise ValueError(
        "content_hash_id is missing from dim_content. "
        f"Available columns: {dim_columns}"
    )

date_candidates = [
    "content_created_at",
    "content_created_date",
    "created_at",
    "created_date",
    "published_at"
]

created_date_column = next(
    (col for col in date_candidates if col in dim_columns),
    None
)

if created_date_column is None:
    raise ValueError(
        "Could not identify the content creation date column. "
        f"Available columns: {dim_columns}"
    )

print("Content creation date field:", created_date_column)


# Step 2: Build five March features.

created_col_sql = '"' + created_date_column.replace('"', '""') + '"'

march_feature_query = f"""
WITH content_dimension AS (
    SELECT
        content_hash_id,
        TRY_CAST({created_col_sql} AS DATE) AS created_date
    FROM {DIM_CONTENT}
),

march_rollup AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(COALESCE(f.gsc_impressions, 0))
            AS impressions_mar,

        SUM(COALESCE(f.gsc_clicks, 0))
            AS clicks_mar,

        SUM(
            CASE
                WHEN f.gsc_avg_position IS NOT NULL
                 AND f.gsc_avg_position > 0
                THEN f.gsc_avg_position
                     * COALESCE(f.gsc_impressions, 0)
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN f.gsc_avg_position IS NOT NULL
                     AND f.gsc_avg_position > 0
                    THEN COALESCE(f.gsc_impressions, 0)
                    ELSE 0
                END
            ), 0
        ) AS avg_position_mar

    FROM {MARCH} AS f

    WHERE f.gsc_data_available IS TRUE

    GROUP BY
        f.client_hash_id,
        f.content_hash_id

)

SELECT
    m.client_hash_id,
    m.content_hash_id,

    m.impressions_mar,
    m.clicks_mar,

    100.0 * m.clicks_mar
        / NULLIF(m.impressions_mar, 0)
        AS ctr_mar_pct,

    m.avg_position_mar,

    CASE
        WHEN d.created_date IS NOT NULL
         AND d.created_date <= DATE '2026-03-31'
        THEN DATE_DIFF(
            'day',
            d.created_date,
            DATE '2026-03-31'
        )
        ELSE NULL
    END AS content_age_days_mar

FROM march_rollup AS m

LEFT JOIN content_dimension AS d
    ON m.content_hash_id = d.content_hash_id

WHERE m.impressions_mar > 0
"""

march_features = con.sql(march_feature_query).df()


# Step 3: Define exactly five predictive features.

FEATURE_COLUMNS = [
    "impressions_mar",
    "clicks_mar",
    "ctr_mar_pct",
    "avg_position_mar",
    "content_age_days_mar"
]

print("\nMARCH FEATURE FRAME")
print("=" * 40)

print("Page-client rows:", len(march_features))
print("Predictive features:", len(FEATURE_COLUMNS))

display(
    march_features[FEATURE_COLUMNS].head(10)
)


# Step 4: Match March features with April outcomes.

feature_frame = march_features.merge(
    april_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    validate="one_to_one"
)

# Target: zero observed April clicks across eligible days.
feature_frame["went_dark_apr"] = (
    feature_frame["clicks_apr"] == 0
).astype("int8")


# Step 5: Display the final feature frame.

print("\nFINAL FEATURE FRAME")
print("=" * 40)

print("Matched page-client rows:", len(feature_frame))

print("\nTarget distribution:")
print(
    feature_frame["went_dark_apr"]
    .value_counts()
    .sort_index()
)

print("\nTarget percentages:")
print(
    (
        feature_frame["went_dark_apr"]
        .value_counts(normalize=True) * 100
    ).round(2)
)

display(
    feature_frame[
        FEATURE_COLUMNS + ["went_dark_apr"]
    ].head(10)
)


# Step 6: Honest quick score using March information.

honest_prediction = (
    feature_frame["clicks_mar"] == 0
).astype("int8")

honest_accuracy = (
    honest_prediction == feature_frame["went_dark_apr"]
).mean()


# Step 7: Deliberately introduce ONE leaking column.

feature_frame["zero_clicks_apr_LEAK"] = (
    feature_frame["clicks_apr"] == 0
).astype("int8")

leaked_accuracy = (
    feature_frame["zero_clicks_apr_LEAK"]
    == feature_frame["went_dark_apr"]
).mean()

print("\nLEAKAGE EXPERIMENT")
print("=" * 40)

print(f"Honest March baseline accuracy: {honest_accuracy:.2%}")
print(f"Leaked accuracy: {leaked_accuracy:.2%}")


# Step 8: Remove the leaking feature and future helper columns.

feature_frame = feature_frame.drop(
    columns=[
        "zero_clicks_apr_LEAK",
        "clicks_apr",
        "gsc_days_apr"
    ]
)

assert "zero_clicks_apr_LEAK" not in feature_frame.columns
assert len(FEATURE_COLUMNS) == 5

print("\nLEAKAGE COLUMN REMOVED")
print("Final feature count:", len(FEATURE_COLUMNS))
print("Honest baseline accuracy retained:",
      f"{honest_accuracy:.2%}")

display(
    feature_frame[
        FEATURE_COLUMNS + ["went_dark_apr"]
    ].head(10)
)



## 4. Data Limits

### Limitation: Unbalanced Panel and Incomplete Future Coverage

The warehouse is an unbalanced panel. Different clients have different amounts of historical data, and GSC availability is not identical across all page-client pairs.

For this initial experiment, I require at least 20 GSC-available days in April before assigning an outcome label.

This means pages with insufficient April observations are excluded from the labelled cohort. The resulting sample may therefore favour pages and clients with more consistent measurement coverage.

Another limitation is that zero observed clicks do not prove that a page needs refreshing or that refreshing it will restore performance. Seasonal changes, demand shifts, consolidation, and measurement gaps may also explain the observed outcome.

The result should be used as decision-support evidence, not as a guarantee of content recovery.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Report the effect of the April coverage requirement

print("DATA COVERAGE LIMITATION SUMMARY")
print("=" * 40)

print(
    "March eligible page-client pairs:",
    len(march_features)
)

print(
    "April pairs with at least 20 measured days:",
    len(april_outcomes)
)

print(
    "Matched and labelled pairs:",
    len(feature_frame)
)

print(
    "March pairs without an eligible April outcome:",
    len(march_features) - len(feature_frame)
)

print(
    "\nFinal predictive feature count:",
    len(FEATURE_COLUMNS)
)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.